In [0]:
%pip install yfinance

In [0]:
%run ./00_config

In [0]:
import pandas as pd, numpy as np, os

ASSETS = pd.DataFrame([
    ("AST-1001","AAPL","Apple","Equity"),
    ("AST-1002","MSFT","Microsoft","Equity"),
    ("AST-1003","NVDA","Nvidia","Equity"),
    ("AST-1004","JPM","JPMorgan Chase","Equity"),
    ("AST-1005","AMZN","Amazon","Equity"),
    ("AST-1006","GOOGL","Alphabet","Equity"),
    ("AST-1007","GS","Goldman Sachs","Equity"),
    ("AST-1008","XOM","Exxon Mobil","Equity"),
    ("AST-1009","JNJ","Johnson & Johnson","Equity"),
    ("AST-1010","SPY","S&P 500 ETF","Equity ETF"),
    ("AST-1011","TLT","20Y Treasury ETF","Fixed Income ETF"),
    ("AST-1012","GLD","Gold ETF","Commodity ETF"),
], columns=["asset_id","ticker","asset_name","asset_class"])

rng = np.random.default_rng(42)          # fixed seed = same data every run
CLIENTS = pd.DataFrame({
    "client_id":    [f"CL-{i:04d}" for i in range(1, 101)],
    "client_name":  [f"Client {i:03d}" for i in range(1, 101)],
    "tier":         rng.choice(["Gold","Platinum","Private"], 100, p=[0.6,0.3,0.1]),
    "risk_profile": rng.choice(["Conservative","Moderate","Aggressive"], 100),
    "updated_at":   "2025-10-01 00:00:00",
})

ASSETS.to_csv(f"{RAW}/assets/assets.csv", index=False)
CLIENTS.to_csv(f"{RAW}/clients/clients.csv", index=False)

In [0]:
import yfinance as yf

START, END = "2025-10-01", "2026-10-16"   # keep END fixed so reruns give identical data
raw = yf.download(ASSETS.ticker.tolist(), start=START, end=END,
                  auto_adjust=False, group_by="ticker", progress=False)

frames = []
for t in ASSETS.ticker:
    d = raw[t].reset_index()
    d["ticker"] = t
    frames.append(d)

px = (pd.concat(frames)
        .dropna(subset=["Close"])
        .rename(columns={"Date":"price_date","Open":"open","High":"high","Low":"low",
                         "Close":"close","Adj Close":"adjusted_close","Volume":"volume"})
        .merge(ASSETS[["asset_id","ticker"]], on="ticker"))
px["price_date"] = pd.to_datetime(px["price_date"]).dt.date
px["updated_at"] = px["price_date"].astype(str) + " 17:00:00"
px = px[["asset_id","price_date","open","high","low","close","adjusted_close","volume","updated_at"]]
print(px.shape, px.price_date.min(), px.price_date.max())

In [0]:
close = px.set_index(["price_date","asset_id"])["close"].to_dict()
dates = sorted(px.price_date.unique())

positions, rows, n = {}, [], 0
for d in dates:
    for _ in range(rng.poisson(24)):                      # ~24 trades a day, ~6,000 total
        c = rng.choice(CLIENTS.client_id)
        a = rng.choice(ASSETS.asset_id)
        if (d, a) not in close:
            continue
        held = positions.get((c, a), 0)
        side = "SELL" if held > 0 and rng.random() < 0.35 else "BUY"
        qty  = int(rng.integers(1, 50)) if side == "BUY" else int(rng.integers(1, held + 1))
        positions[(c, a)] = held + qty if side == "BUY" else held - qty
        n += 1
        rows.append(dict(
            trade_id=f"TRD-{n:06d}", client_id=c, asset_id=a, trade_date=d, side=side,
            quantity=qty, unit_price=round(close[(d, a)] * (1 + rng.uniform(-0.005, 0.005)), 4),
            updated_at=f"{d} 18:00:00"))

trades = pd.DataFrame(rows)
trades["landing_date"] = trades["trade_date"]   # which day's file each row goes into
print(len(trades), trades.side.value_counts().to_dict())

In [0]:
test_day1, test_day2 = dates[-2], dates[-1]

# 3 exact duplicates inside test day 1's file
dupes = trades[trades.trade_date == test_day1].head(3)

# a price correction for one test-day-1 BUY, arriving in test day 2's file with a later updated_at
corr = trades[(trades.trade_date == test_day1) & (trades.side == "BUY")].head(1).copy()
corr["unit_price"]   = (corr["unit_price"] * 1.01).round(4)
corr["updated_at"]   = f"{test_day2} 09:00:00"
corr["landing_date"] = test_day2

trades_out = pd.concat([trades, dupes, corr], ignore_index=True)
print("correction:", corr.trade_id.iloc[0])     # note this ID for Chunk 5

In [0]:
def land_day(d):
    for name, df in [("prices", px[px.price_date == d]),
                     ("trades", trades_out[trades_out.landing_date == d].drop(columns="landing_date"))]:
        folder = f"{RAW}/{name}/date={d}"
        os.makedirs(folder, exist_ok=True)
        df.to_csv(f"{folder}/{name}_{d:%Y%m%d}.csv", index=False)

#for d in dates[:-2]:          # backfill everything except the two test days
#    land_day(d)

In [0]:
# no client ever goes below zero shares, in trade order
signed = np.where(trades.side == "BUY", trades.quantity, -trades.quantity)
running = trades.assign(s=signed).groupby(["client_id","asset_id"])["s"].cumsum()
assert (running >= 0).all(), "oversell found"

# trade prices sit within 0.5% of the close
chk = trades.merge(px, left_on=["trade_date","asset_id"], right_on=["price_date","asset_id"])
print("max deviation:", ((chk.unit_price / chk.close) - 1).abs().max())

display(dbutils.fs.ls(f"{RAW}/trades")[-1])

In [0]:
display(test_day1)

In [0]:
   land_day(test_day1)
   n_day1 = (trades.trade_date == test_day1).sum()
   pairs_day1 = trades[trades.trade_date == test_day1][["client_id", "asset_id"]].drop_duplicates().shape[0]
   print("test day 1:", test_day1, "| trades:", n_day1, "| pairs:", pairs_day1)

In [0]:
land_day(test_day2)